# Task 3: Broadcast Join Experiment for Query 2



## Connect to Spark

In [9]:
import sys

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import execute_registered_query
from analytics.source_tables import register_broadcast_join_sources

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-BroadcastJoin-Query3')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark

In [13]:
spark.catalog.clearCache()
spark.read.format("delta") \
    .load("s3a://delta-data/gold/gold_integrated_taxi_trips") \
    .createOrReplaceTempView("integrated_taxi_trips")
spark.read.format("delta") \
    .load("s3a://delta-data/silver/silver_weather_conditions") \
    .createOrReplaceTempView("weather_conditions")

26/09/20 13:59:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Isolate the join strategy

- Automatic broadcasting is disabled
- AQE is also disabled so it cannot change the join at [runtime](https://spark.apache.org/docs/4.0.4/sql-performance-tuning.html#adaptive-query-execution).
- The forced query uses a broadcast hint.

In [14]:
JOIN_CONFIGS = {
    'spark.sql.adaptive.enabled': 'false',
    'spark.sql.autoBroadcastJoinThreshold': '-1',
}
for key, value in JOIN_CONFIGS.items():
    spark.conf.set(key, value)

JOIN_CONFIGS

{'spark.sql.adaptive.enabled': 'false',
 'spark.sql.autoBroadcastJoinThreshold': '-1'}

## Measure normal and broadcast joins

## Normal-join physical plan

This indicates normal join was used: `SortMergeJoin`.

In [15]:
NORMAL_QUERY = 'query_02'
BROADCAST_QUERY = 'query_02_bj'
REPETITIONS = 9

normal_results = measure_registered_query(spark, NORMAL_QUERY, REPETITIONS)
broadcast_results = measure_registered_query(spark, BROADCAST_QUERY, REPETITIONS)

assert rows_match(
    normal_results['rows'],
    broadcast_results['rows'],
), 'Normal and broadcast results differ.'

print('Normal and broadcast results match within float tolerance.')

Normal and broadcast results match within float tolerance.


In [16]:
execute_registered_query(spark, NORMAL_QUERY).explain('formatted')

== Physical Plan ==
* Sort (18)
+- Exchange (17)
   +- * HashAggregate (16)
      +- Exchange (15)
         +- * HashAggregate (14)
            +- * Project (13)
               +- * SortMergeJoin Inner (12)
                  :- * Sort (6)
                  :  +- Exchange (5)
                  :     +- * Project (4)
                  :        +- * Filter (3)
                  :           +- * ColumnarToRow (2)
                  :              +- Scan parquet  (1)
                  +- * Sort (11)
                     +- Exchange (10)
                        +- * Filter (9)
                           +- * ColumnarToRow (8)
                              +- Scan parquet  (7)


(1) Scan parquet 
Output [3]: [trip_distance#115, coco_pu#155, pickup_month#130]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
PushedFilters: [IsNotNull(coco_pu)]
ReadSchema: struct<trip_distance:double,coco_pu:int>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [tri

## Broadcast-join physical plan

- These indicate broadcast join was used: `BroadcastHashJoin` and `BroadcastExchange`.

In [17]:
execute_registered_query(spark, BROADCAST_QUERY).explain('formatted')

== Physical Plan ==
* Sort (15)
+- Exchange (14)
   +- * HashAggregate (13)
      +- Exchange (12)
         +- * HashAggregate (11)
            +- * Project (10)
               +- * BroadcastHashJoin Inner BuildRight (9)
                  :- * Project (4)
                  :  +- * Filter (3)
                  :     +- * ColumnarToRow (2)
                  :        +- Scan parquet  (1)
                  +- BroadcastExchange (8)
                     +- * Filter (7)
                        +- * ColumnarToRow (6)
                           +- Scan parquet  (5)


(1) Scan parquet 
Output [3]: [trip_distance#115, coco_pu#155, pickup_month#130]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
PushedFilters: [IsNotNull(coco_pu)]
ReadSchema: struct<trip_distance:double,coco_pu:int>

(2) ColumnarToRow [codegen id : 2]
Input [3]: [trip_distance#115, coco_pu#155, pickup_month#130]

(3) Filter [codegen id : 2]
Input [3]: [trip_distance#115, coco_pu#1

## Benchmark summary

In [18]:
summary = {
    'normal_samples_seconds': [round(value, 3) for value in normal_results['samples_seconds']],
    'normal_median_seconds': round(normal_results['median_seconds'], 3),
    'broadcast_samples_seconds': [round(value, 3) for value in broadcast_results['samples_seconds']],
    'broadcast_median_seconds': round(broadcast_results['median_seconds'], 3),
    'broadcast_speedup_factor': round(normal_results['median_seconds'] / broadcast_results['median_seconds'],2),
}
summary

{'normal_samples_seconds': [5.125,
  5.187,
  4.557,
  4.274,
  5.525,
  4.366,
  3.85,
  4.027,
  3.688],
 'normal_median_seconds': 4.366,
 'broadcast_samples_seconds': [2.199,
  1.754,
  2.258,
  1.645,
  1.547,
  1.626,
  1.434,
  1.433,
  1.204],
 'broadcast_median_seconds': 1.626,
 'broadcast_speedup_factor': 2.68}

## Cleanup

In [19]:
spark.catalog.clearCache()
spark.stop()

26/09/20 14:03:58 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-71120cfa-a283-4e9c-8da3-a30bc4ea96a5/pyspark-e4dbe0b2-8e13-4228-9ca5-3f4e75b331ad. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-71120cfa-a283-4e9c-8da3-a30bc4ea96a5/pyspark-e4dbe0b2-8e13-4228-9ca5-3f4e75b331ad
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:199)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:116)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:94)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1048)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:70)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun